# Causal Inference Crash Course:
## Part 10 — Conformal Inference
Julian Hsu

## Causal Inference Series
1. Foundations
2. Causal Models
3. Inference
4. Best Practices
5. Heterogeneous Treatment Effects
6. Panel Models
7. Regression Discontinuity
8. Surrogate Models
9. Arguable Validation
10. **Conformal Inference  ← this deck**

## Overview
* **Conformal inference** turns *any* model's point prediction into a **set**
  with a **finite-sample, distribution-free coverage guarantee** — it needs only
  **exchangeability**, not a distribution and not a correctly specified model.
* Why we care: honest uncertainty for the black-box predictors we actually use —
  and, the payoff, **valid causal inference with a single treated unit**
  (synthetic control), where classical standard errors don't apply.
* We cover:
    * the idea: **split conformal** and its coverage guarantee
    * the one assumption — **exchangeability** — and what breaks it
    * two fixes when it fails: **moving blocks** and **weighting**
    * the causal payoff: conformal inference for **synthetic controls**
* Assumes the earlier **Inference** and **Panel Models** decks.

# The Idea

## What we want
* Classical prediction intervals lean on a **model being right** — Wald needs a
  correct parametric form and large samples; Bayes needs a correct likelihood.
* **Conformal** asks for far less: a set $\mathcal C(x)$ with
$$ \mathbb P\big(Y_{n+1}\in\mathcal C(X_{n+1})\big)\ \ge\ 1-\alpha $$
  that holds in **finite samples**, for **any** data distribution, wrapped around
  **any** predictor.
* Running example: predict a new customer's next-quarter purchases and give an
  interval that really covers 90% of the time — whatever model produced the point
  prediction.

## The one assumption: exchangeability
* The observations $(X_1,Y_1),\dots,(X_{n+1},Y_{n+1})$ are **exchangeable**: their
  joint distribution is unchanged by reordering them. (i.i.d. is the familiar
  special case.)
* That is the *whole* assumption — no distributional form, no correctly specified
  model. Everything downstream is a consequence of it.
* It is also exactly what **fails** for time series and panels — we will come
  back to that.

## Conformity scores
* Pick a **nonconformity score** $s(x,y)$ — "how strange is this pair, to the
  model?" Larger means a worse fit.
* Regression default: the absolute residual of any fitted $\hat f$,
$$ s(x,y)=|y-\hat f(x)|. $$
* The guarantee holds for **any** score and **any** model; a *good* score only
  makes the resulting set **tighter**, never invalid.

## Split conformal — the recipe
For miscoverage level $\alpha$, split the data into a **training** fold and a
**calibration** fold of size $n$:
1. **Fit** $\hat f$ on the training fold only.
2. **Score** the calibration set: $s_i = s(X_i, Y_i)$, $i=1,\dots,n$.
3. **Quantile:** $\hat q = s_{(k)}$, the $k$-th smallest score, with
   $k=\lceil (n+1)(1-\alpha)\rceil$.
4. **Predict:** $\mathcal C(x)=\{\,y : s(x,y)\le \hat q\,\}$.
![Image](Figures/Conformal_Figure_0.png)

## The coverage guarantee
* **Theorem** (Vovk et al. 2005; Lei et al. 2018). If the data are
  **exchangeable**, split conformal satisfies
$$ 1-\alpha\ \le\ \mathbb P\big(Y_{n+1}\in\mathcal C(X_{n+1})\big)\ \le\ 1-\alpha+\tfrac{1}{n+1}. $$
* **Finite-sample, distribution-free, model-free.** The upper bound needs only
  that scores have no ties.
* The model can be **arbitrarily wrong** — coverage still holds. A bad model
  simply yields an honestly **wide** set.

## Why it works
* Insert the unseen test score as an $(n{+}1)$-th point. By exchangeability its
  **rank is uniform** on $\{1,\dots,n+1\}$, so it exceeds the $k$-th calibration
  value only about $\alpha$ of the time — which is exactly the miscoverage event.
* So coverage is a **property of the procedure**, not of any one interval: over
  repeated draws it follows a $\text{Beta}(k,\,n{+}1{-}k)$ law centered at $1-\alpha$.
![Image](Figures/Conformal_Figure_1.png)

## The p-value view
* Equivalent framing (Shafer & Vovk 2008): conformal is **hypothesis testing
  turned into prediction**. For each candidate label $y$,
$$ p(y)=\frac{1+\#\{\,i:s_i\ge s(x,y)\,\}}{n+1},\qquad
   \mathcal C(x)=\{\,y : p(y)>\alpha\,\}. $$
* Keep the labels you **cannot reject** at level $\alpha$.
* This is the view we will use for synthetic control — where the "label" becomes
  a hypothesized treatment effect.

## Adaptivity — and its limit
* The guarantee is **marginal**: it averages over draws. It is **not
  conditional** — some regions can be under-covered and others over-covered, and
  exact conditional coverage is impossible distribution-free.
* Fix with a smarter *score*, not a new guarantee: normalize by a local scale
  $s=|y-\hat f(x)|/\hat\sigma(x)$, or use **conformalized quantile regression**
  (Romano, Patterson & Candès 2019).
![Image](Figures/Conformal_Figure_2.png)

# When Exchangeability Fails

## Time and treatment break exchangeability
* Panels and time series are **serially dependent** and often **drift** — reorder
  them and the joint law changes, so exchangeability fails.
* Worse for causal work: the null holds only **before** treatment, and the
  post-period is exactly what we suspect differs.
* Plain conformal then **under-covers** — its interval is calibrated on a stale
  past that no longer resembles the present.
![Image](Figures/Conformal_Figure_3.png)

## Fix A — permute blocks, not points
* Keep **contiguous blocks** of residuals intact and permute the block
  **positions** — the moving-block idea borrowed from the block bootstrap.
* Local dependence rides along *inside* each block; only its placement is
  randomized, so the null datasets keep the real time structure.
* **Exact** under exchangeability, **approximately valid** under stationarity and
  weak dependence — the block length must exceed the dependence range
  (Chernozhukov, Wüthrich & Zhu 2018, 2021).

## Fix B — weight what belongs
* **Nonexchangeable conformal** (Barber, Candès, Ramdas & Tibshirani 2023):
  attach fixed weights $w_i$ and take **weighted quantiles** of the scores — for
  time series, weight **recent** points more.
* The guarantee degrades gracefully:
$$ \mathbb P\big(Y_{n+1}\in\mathcal C(X_{n+1})\big)\ \ge\ 1-\alpha-\Delta, $$
  where $\Delta$ is a weighted **distance from exchangeability**; it vanishes when
  the up-weighted points really do resemble the target.
* The weights must be **fixed a priori** — not learned from the scores.

## Two relaxations of one idea
* **Blocks** — the *discrete* relaxation: glue dependent points together so a
  permutation can't shatter the local structure.
* **Weights** — the *continuous* relaxation: down-weight the points that don't
  belong with the target.
* Both **restore an approximate symmetry** the raw data lacks — just enough for
  the uniform-rank argument to (nearly) go through.

# Conformal Inference for Synthetic Controls

## The CWZ recast
* **Chernozhukov, Wüthrich & Zhu (2021)** glue two familiar problems together:
  causal inference $=$ **counterfactual prediction** $+$ **a test for a
  structural break**.
* Predict the treated unit's *untreated* path (synthetic control, DiD, factor
  models…), then ask whether the **post-treatment residuals break** from the
  pre-treatment ones.
* The pitch: **inference that doesn't care how good your synthetic control is** —
  validity comes from the permutation, not from a correct model.

## Setup — we predict the counterfactual
* One treated unit $Y_{1t}$, a donor pool $Y_{0t}$, pre-periods $t\le T_0$ and
  post-periods $t>T_0$. Fit a **synthetic control** $\hat Y_{1t}(0)$ (from the
  Panel Models deck).
* The **gap** $\hat u_t = Y_{1t}-\hat Y_{1t}(0)$ is the residual. Under *no
  effect*, that residual process has **no break** at $T_0$.
* The residuals over time are serially dependent — so this is exactly the
  non-exchangeable case, handled by **moving-block** permutations.

## Step 1 — impose the null, read the residuals
* Test a sharp null on the post-period effect — here a constant effect
  $H_0:\tau_t=\theta$. Subtract it from the **post outcomes only**, refit over the
  **whole timeline**, and take residuals for every $t$:
$$ \tilde Y_{1t}=Y_{1t}-\theta\cdot\mathbf 1\{t>T_0\},\qquad
   \hat u_t=\tilde Y_{1t}-\hat Y_{1t}(0) $$
![Image](Figures/Conformal_Figure_4.png)

## Step 2 — a statistic and a moving-block permutation
* Summarize the post-period residuals with a single number ($q=1$ = mean
  absolute residual):
$$ S_q(\hat u)=\Big(\tfrac{1}{T_{post}}\textstyle\sum_{t>T_0}|\hat u_t|^{\,q}\Big)^{1/q} $$
* Compare $S_q$ to its distribution under **moving-block permutations** of the
  residual sequence — the serial-dependence-robust scheme from the last section.
* `panelib`'s `conformal_inf` implements exactly this.

## Step 3 — the permutation p-value
$$ p=\frac{1}{|\Pi|}\sum_{\pi\in\Pi}\mathbf 1\{\,S_q(\hat u_\pi)\ge S_q(\hat u)\,\},
   \qquad \text{reject } H_0 \text{ if } p\le\alpha. $$
![Image](Figures/Conformal_Figure_5.png)
* **Careful:** this permutes residuals **across time** for one treated unit — not
  the Abadie placebo test, which permutes treatment **across units**.

## Step 4 — confidence interval by test inversion
* Repeat the test over a grid of candidate effects $\theta$ and keep every value
  you **cannot reject**:
$$ \text{CI}_{1-\alpha}=\{\,\theta : p(\theta)>\alpha\,\} $$
![Image](Figures/Conformal_Figure_6.png)
* The point estimate sits where $p(\theta)$ peaks; the CI is the set of effects
  consistent with "no break."

## Why this matters
* Works with **one treated unit and few pre-periods** — and is **exact** under
  exchangeability, **robust** to serial dependence and to a **misspecified**
  counterfactual (validity comes from residual symmetry, not a correct model).
* The fine print: under dependence the guarantee is approximate, and it needs the
  counterfactual's **error to be stable over time** — a model whose bias *trends*
  into the post-period breaks a symmetry no permutation can restore.
* Resolution matters too: with very few post-periods the permutation p-value is
  coarse — in the Panel deck's simulations (2 post-periods) coverage ran
  **80–85%**, not 95%.
* Contrast Abadie placebo inference: its p-value resolution is only $1/(J+1)$ and
  it needs a large donor pool.
* This is the inference the **Panel Models** deck used to put DiD, SC, and SDID
  on the same footing.

## Conclusion
* Conformal inference buys an **honest, finite-sample guarantee** from **one
  assumption you can actually reason about** — exchangeability — and degrades
  gracefully when it fails.
* Reach for it to **wrap black-box predictors**, and — the causal payoff — to do
  inference where classical SEs can't: **a single treated unit over time**.
* The price: coverage is **marginal**, not conditional; and under dependence you
  must **block** or **weight** to keep the guarantee.
* Start from the guarantee you can defend, then pick the score and the
  permutation that fit your data.

# Appendix

## Appendix: split vs. full conformal
* **Split (inductive)** conformal — the recipe here — spends part of the data on
  a calibration fold. Cheap (one fit), but the interval is random in that split
  and a bit wider.
* **Full** conformal refits for every candidate label — exact but expensive;
  **CV+ / jackknife+** (Barber et al. 2021) recover most of the efficiency
  without a held-out split.
* Same guarantee; the choice is **data efficiency vs. compute**.

## Appendix: the coverage law
* Conditional on the calibration draw, the coverage of split conformal is a
  **random variable**:
$$ \mathbb P\big(Y_{n+1}\in\mathcal C\mid \text{calibration}\big)\ \sim\
   \text{Beta}\big(k,\,n+1-k\big),\quad k=\lceil(n+1)(1-\alpha)\rceil. $$
* Its mean is $\ge 1-\alpha$; the spread shrinks like $1/\sqrt n$ — the middle
  histogram made this concrete.
* The upper bound $1-\alpha+\tfrac1{n+1}$ needs the scores to have **no ties**
  (continuous scores, or a random tie-break).

## Appendix: another route — prediction intervals
* Conformal is not the only distribution-aware option for synthetic control.
* **Cattaneo, Feng & Titiunik (2021)** build **prediction intervals** that
  separate two error sources: the in-sample **weight-construction** error and the
  out-of-sample **stochastic** error, with finite-sample guarantees.
* Rule of thumb: **conformal** tests a sharp null by permutation and inverts to a
  CI; **prediction intervals** model the two error sources directly. Different
  assumptions, complementary diagnostics.

## Appendix: conformal for treatment effects, beyond panels
* **Individual treatment effects** — Lei & Candès (2021): conformal intervals for
  $Y(1)-Y(0)$ under unconfoundedness, using *weighted* conformal to bridge the
  treated/control covariate shift. The natural companion to the HTE deck.
* **Covariate shift** — Tibshirani, Barber, Candès, Ramdas (2019): weighted
  conformal stays valid when the test point comes from a different covariate
  distribution with a known (or estimated) likelihood ratio.
* **Drift in production** — Gibbs & Candès (2021): *adaptive* conformal updates
  the level $\alpha_t$ online to keep realized coverage on target under
  distribution shift — the monitoring use case for deployed models.

## Literature Review of Related Papers
* **Conformal prediction — foundations**
    * Vovk, Gammerman, Shafer (2005). *Algorithmic Learning in a Random World.*
      Springer — the founding monograph. [link](https://alrw.net/)
    * Shafer & Vovk (2008). "A Tutorial on Conformal Prediction." *JMLR* 9,
      371–421. [link](https://jmlr.org/papers/v9/shafer08a.html)
    * Lei, G'Sell, Rinaldo, Tibshirani, Wasserman (2018). "Distribution-Free
      Predictive Inference for Regression." *JASA* 113(523). [link](https://arxiv.org/abs/1604.04173)
    * Angelopoulos & Bates (2023). "A Gentle Introduction to Conformal Prediction
      and Distribution-Free Uncertainty Quantification." *Found. & Trends ML.*
      [link](https://arxiv.org/abs/2107.07511)
* **Adaptivity & beyond exchangeability**
    * Romano, Patterson, Candès (2019). "Conformalized Quantile Regression."
      *NeurIPS.* [link](https://arxiv.org/abs/1905.03222)
    * Barber, Candès, Ramdas, Tibshirani (2021). "Predictive Inference with the
      Jackknife+." *Ann. Statist.* 49(1). [link](https://arxiv.org/abs/1905.02928)
    * Barber, Candès, Ramdas, Tibshirani (2023). "Conformal Prediction Beyond
      Exchangeability." *Ann. Statist.* 51(2), 816–845. [link](https://arxiv.org/abs/2202.13415)
    * Tibshirani, Barber, Candès, Ramdas (2019). "Conformal Prediction Under
      Covariate Shift." *NeurIPS.* [link](https://arxiv.org/abs/1904.06019)
    * Gibbs & Candès (2021). "Adaptive Conformal Inference Under Distribution
      Shift." *NeurIPS.* [link](https://arxiv.org/abs/2106.00170)

## Literature Review (continued)
* **Conformal inference for causal panels / synthetic control**
    * Chernozhukov, Wüthrich, Zhu (2018). "Exact and Robust Conformal Inference
      Methods for Predictive ML with Dependent Data." *COLT (PMLR 75).*
      [link](https://proceedings.mlr.press/v75/chernozhukov18a.html)
    * Chernozhukov, Wüthrich, Zhu (2021). "An Exact and Robust Conformal Inference
      Method for Counterfactual and Synthetic Controls." *JASA* 116(536),
      1849–1864. [link](https://www.tandfonline.com/doi/full/10.1080/01621459.2021.1920957)
    * Cattaneo, Feng, Titiunik (2021). "Prediction Intervals for Synthetic Control
      Methods." *JASA* 116(536), 1865–1880. [link](https://www.tandfonline.com/doi/abs/10.1080/01621459.2021.1979561)
    * Firpo & Possebom (2018). "Synthetic Control Method: Inference, Sensitivity
      Analysis and Confidence Sets." *J. Causal Inference* 6(2). [link](https://doi.org/10.1515/jci-2016-0026)
    * Lei & Candès (2021). "Conformal Inference of Counterfactuals and Individual
      Treatment Effects." *JRSS-B* 83(5). [link](https://arxiv.org/abs/2006.06138)
    * Abadie, Diamond, Hainmueller (2010). "Synthetic Control Methods for
      Comparative Case Studies." *JASA* 105(490). [link](https://www.tandfonline.com/doi/abs/10.1198/jasa.2009.ap08746)
* **Companion code:** `panelib.py` (`conformal_inf`) in
  [statanomics](https://github.com/shoepaladin/statanomics/blob/main/workingcode/panelmodels/panelib.py).

In [ ]:
# --- Figure generation (run this notebook to regenerate Figures/Conformal_Figure_*.png) ---
# General conformal figures (0-3) come from simple regression simulations; the
# synthetic-control application (4-6) is produced from a real panelib SC-ADH fit
# via the conformal_inf API. panelib is the canonical panel library in the
# *statanomics* repo; we import it from a sibling clone (shallow-cloning it if
# absent) so the code is never duplicated.
import os, sys, io, subprocess, warnings
from contextlib import redirect_stdout
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def _ensure_panelib():
    here = os.getcwd()
    cands = [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here)),
             os.path.expanduser("~")]
    for base in cands:
        p = os.path.join(base, "statanomics", "workingcode", "panelmodels")
        if os.path.exists(os.path.join(p, "panelib.py")):
            return p
    dest = os.path.join(os.path.dirname(here), "statanomics")
    if not os.path.exists(dest):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/shoepaladin/statanomics.git", dest],
                       check=True)
    return os.path.join(dest, "workingcode", "panelmodels")

sys.path.insert(0, _ensure_panelib())
from panelib import sc, dgp, conformal_inf

ACCENT, ACCENT2, CTRLC = "#0e7490", "#0f766e", "#94a3b8"
INK, MUTED, LINE = "#17222e", "#5b6b7a", "#cbd5e1"
RED, AMBER, VIOLET = "#b91c1c", "#d97706", "#6d28d9"
OBS = "#334155"
FIGDIR = os.path.join(os.getcwd(), "Figures")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.grid": True,
                     "grid.alpha": .25, "axes.edgecolor": "#cbd5e1",
                     "axes.linewidth": .8})
DD = {"treatment": "treated", "date": "time", "post": "post",
      "unitid": "unit_id", "outcome": "y"}


def _save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, name), bbox_inches="tight")
    plt.close(fig)

In [ ]:
def _f_true(x):
    return 2.0 + 1.3 * x + 1.5 * np.sin(1.3 * x)


def _make(n, sd=1.0, rng=None, hetero=False):
    rng = rng or np.random.default_rng(0)
    x = rng.uniform(0, 10, n)
    s = sd * (0.3 + 0.35 * x) if hetero else sd
    y = _f_true(x) + rng.normal(0, 1, n) * s
    return x, y


def _fit_poly(x, y, deg=3):
    c = np.polyfit(x, y, deg)
    return lambda xx: np.polyval(c, xx)


# ---------------------------------------------------------------- Figure 0
# Split-conformal calibration: score histogram with the (1-alpha) quantile cut.
def fig0():
    rng = np.random.default_rng(1); alpha = 0.1
    xt, yt = _make(600, rng=rng); f = _fit_poly(xt, yt)
    xc, yc = _make(300, rng=rng)
    scores = np.abs(yc - f(xc)); n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    qhat = np.sort(scores)[k - 1]
    fig, ax = plt.subplots(figsize=(8.4, 3.9))
    ax.hist(scores, bins=30, color=CTRLC, edgecolor="white", alpha=.9)
    ax.axvline(qhat, color=RED, lw=2.4,
               label=fr"$\hat q = s_{{(k)}}$,  $k=\lceil(n+1)(1-\alpha)\rceil$   = {qhat:.2f}")
    ax.axvspan(qhat, scores.max() * 1.02, color=RED, alpha=.06)
    ax.annotate(r"$\approx\alpha$ of mass", xy=(qhat, ax.get_ylim()[1] * .6),
                xytext=(8, 0), textcoords="offset points", color=RED, fontsize=9)
    ax.set_xlabel(r"calibration nonconformity score  $s_i=|y_i-\hat f(x_i)|$")
    ax.set_ylabel("count"); ax.grid(axis="x")
    ax.set_title(f"Split conformal: threshold at the (1−α) quantile  (n={n}, α=0.1)")
    ax.legend(loc="upper right", fontsize=9, framealpha=.9)
    _save(fig, "Conformal_Figure_0.png")


# ---------------------------------------------------------------- Figure 1
# Coverage validation: empirical coverage over many splits ~ Beta(k, n+1-k).
def fig1():
    rng = np.random.default_rng(2); alpha = 0.1
    xt, yt = _make(600, rng=rng); f = _fit_poly(xt, yt)
    n_cal, n_test, R = 200, 500, 2000
    k = int(np.ceil((n_cal + 1) * (1 - alpha)))
    covs = []
    for _ in range(R):
        xc, yc = _make(n_cal, rng=rng); xe, ye = _make(n_test, rng=rng)
        qhat = np.sort(np.abs(yc - f(xc)))[k - 1]
        covs.append(np.mean(np.abs(ye - f(xe)) <= qhat))
    covs = np.array(covs)
    fig, ax = plt.subplots(figsize=(8.4, 3.9))
    ax.hist(covs, bins=30, density=True, color=ACCENT, edgecolor="white", alpha=.85,
            label=f"empirical coverage over {R} splits")
    gg = np.linspace(covs.min(), covs.max(), 200)
    ax.plot(gg, stats.beta(k, n_cal + 1 - k).pdf(gg), color=INK, lw=2,
            label=r"Beta$(k,\,n{+}1{-}k)$ law")
    ax.axvline(1 - alpha, color=RED, lw=2, ls="--", label=r"target  $1-\alpha=0.90$")
    ax.set_xlabel("empirical coverage"); ax.set_ylabel("density"); ax.grid(axis="x")
    ax.set_title("Coverage is a random quantity, centered at the guarantee")
    ax.legend(loc="upper left", fontsize=8.5, framealpha=.9)
    _save(fig, "Conformal_Figure_1.png")


# ---------------------------------------------------------------- Figure 2
# Adaptivity: constant-width vs normalized (locally adaptive) conformal bands.
def fig2():
    rng = np.random.default_rng(5); alpha = 0.1
    xt, yt = _make(700, rng=rng, hetero=True); f = _fit_poly(xt, yt)
    xc, yc = _make(400, rng=rng, hetero=True)
    # local scale model sigma_hat(x): fit |resid| on x
    res_c = np.abs(yc - f(xc))
    cs = np.polyfit(xc, res_c, 2); sig = lambda xx: np.clip(np.polyval(cs, xx), 0.2, None)
    n = len(yc); k = int(np.ceil((n + 1) * (1 - alpha)))
    q_abs = np.sort(res_c)[k - 1]
    q_norm = np.sort(res_c / sig(xc))[k - 1]
    xe, ye = _make(1500, rng=rng, hetero=True)
    cov_c = np.mean(np.abs(ye - f(xe)) <= q_abs)
    cov_a = np.mean(np.abs(ye - f(xe)) <= q_norm * sig(xe))
    xg = np.linspace(0, 10, 200)
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.0), sharey=True)
    for ax, (lab, lo, hi, cov) in zip(axes, [
            ("Constant width  $s=|y-\\hat f|$", f(xg) - q_abs, f(xg) + q_abs, cov_c),
            ("Adaptive  $s=|y-\\hat f|/\\hat\\sigma(x)$",
             f(xg) - q_norm * sig(xg), f(xg) + q_norm * sig(xg), cov_a)]):
        ax.scatter(xe, ye, s=6, color=CTRLC, alpha=.4)
        ax.fill_between(xg, lo, hi, color=ACCENT, alpha=.2)
        ax.plot(xg, f(xg), color=ACCENT2, lw=2)
        ax.set_xlabel("x"); ax.set_title(f"{lab}\ncoverage {cov:.2f}", fontsize=10.5)
    axes[0].set_ylabel("y")
    fig.suptitle("Same 90% marginal coverage, very different local behavior", y=1.02)
    _save(fig, "Conformal_Figure_2.png")


# ---------------------------------------------------------------- Figure 3
# Exchangeability fails under drift: plain conformal under-covers; a recent-
# weighted (nonexchangeable) variant restores coverage. Rolling coverage vs time.
def fig3():
    rng = np.random.default_rng(9); alpha = 0.1
    T = 600
    t = np.arange(T)
    scale = 0.6 + 2.2 * (t / T) ** 2               # volatility drifts up over time
    y = _f_true(t / 60.0) + rng.normal(0, 1, T) * scale
    f = _fit_poly(t / 60.0, y, deg=4)
    resid = np.abs(y - f(t / 60.0))
    k_frac = 1 - alpha
    win = 100
    roll_plain, roll_wt, centers = [], [], []
    for e in range(win, T):
        centers.append(e)
        cal_all = resid[:e]                         # plain: all history, equal weight
        q_plain = np.sort(cal_all)[int(np.ceil((len(cal_all) + 1) * k_frac)) - 1]
        cal_rec = resid[max(0, e - win):e]          # weighted: recent window only
        q_wt = np.sort(cal_rec)[int(np.ceil((len(cal_rec) + 1) * k_frac)) - 1]
        # coverage on the next block of points
        blk = slice(e, min(e + 20, T))
        roll_plain.append(np.mean(resid[blk] <= q_plain))
        roll_wt.append(np.mean(resid[blk] <= q_wt))
    centers = np.array(centers)
    fig, ax = plt.subplots(figsize=(8.6, 3.9))
    ax.plot(centers, pd.Series(roll_plain).rolling(30, min_periods=1).mean(),
            color=RED, lw=2, label="plain conformal (all history, equal weight)")
    ax.plot(centers, pd.Series(roll_wt).rolling(30, min_periods=1).mean(),
            color=ACCENT, lw=2, label="recent-weighted (nonexchangeable)")
    ax.axhline(1 - alpha, color=MUTED, ls="--", lw=1.4, label="target 0.90")
    ax.set_xlabel("time  (volatility drifts upward →)")
    ax.set_ylabel("rolling coverage"); ax.set_ylim(0.5, 1.02)
    ax.set_title("Under drift, exchangeability breaks and plain conformal under-covers")
    ax.legend(loc="lower left", fontsize=8.5, framealpha=.9)
    _save(fig, "Conformal_Figure_3.png")


# ============================ SYNTHETIC-CONTROL APPLICATION ===================
T_PRE_SC, T_POST_SC = 15, 6


def _fit_sc():
    df, att = dgp.simulate_panel(seed=7, T_pre=T_PRE_SC, T_post=T_POST_SC,
                                 N_control=30, noise_sd=0.3, att_pct=0.7,
                                 rho=0.8, sigma_lambda=0.0)
    tid = df.loc[df.treated == 1, "unit_id"].iloc[0]
    with redirect_stdout(io.StringIO()):
        r = sc.sc_model(model_name="adh", data=df, data_dict=DD,
                        inference={"alpha": 0.05, "theta_grid": np.arange(-6, 6, 0.05)})
    pe = r["predict_est"]
    obs = pe[tid].to_numpy(); cf = pe[f"{tid}_est"].to_numpy()
    perm, lengths = conformal_inf.time_block_permutation(
        data=df[df.treated == 1], time_unit="time", post="post")
    return df, tid, obs, cf, perm, lengths, att


def fig4(fit=None):                                   # residuals under the null
    df, tid, obs, cf, perm, lengths, att = fit or _fit_sc()
    t = np.arange(len(obs)); resid = obs - cf
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.8))
    ax = axes[0]
    ax.plot(t, obs, color=ACCENT, lw=2.2, label="treated (observed)")
    ax.plot(t, cf, "--", color=ACCENT2, lw=2.0, label="synthetic control")
    ax.axvspan(T_PRE_SC - .5, t[-1], color=RED, alpha=.06)
    ax.axvline(T_PRE_SC - .5, color=MUTED, ls="--", lw=1.1)
    ax.set_xlabel("time"); ax.set_ylabel("outcome  Y")
    ax.set_title("A synthetic control that fits the pre-period")
    ax.legend(loc="upper left", fontsize=8.5, framealpha=.9)
    ax = axes[1]
    ax.axhline(0, color=INK, lw=1)
    ax.scatter(t[:T_PRE_SC], resid[:T_PRE_SC], color=MUTED, s=26, zorder=3,
               label="pre  (should be ~0)")
    ax.scatter(t[T_PRE_SC:], resid[T_PRE_SC:], color=ACCENT, s=34, zorder=3,
               label="post  (carries the effect)")
    ax.vlines(t, 0, resid, color=CTRLC, lw=1)
    ax.axvspan(T_PRE_SC - .5, t[-1], color=RED, alpha=.06)
    ax.axvline(T_PRE_SC - .5, color=MUTED, ls="--", lw=1.1)
    ax.set_xlabel("time"); ax.set_ylabel(r"residual  $\hat u_t = Y_t - \hat Y_t(0)$")
    ax.set_title("Residuals under the no-effect null")
    ax.legend(loc="upper left", fontsize=8.5, framealpha=.9)
    _save(fig, "Conformal_Figure_4.png")


def fig5(fit=None):                                   # permutation null dist
    df, tid, obs, cf, perm, lengths, att = fit or _fit_sc()
    full_resid = np.abs(obs - cf)
    S_obs = conformal_inf.test_statS(1, lengths, full_resid[-T_POST_SC:])
    S_perm = np.array([conformal_inf.test_statS(1, lengths,
                       full_resid[np.array(p)][-T_POST_SC:]) for p in perm])
    pval = 1 - np.mean(S_perm < S_obs)
    fig, ax = plt.subplots(figsize=(8.4, 3.9))
    ax.hist(S_perm, bins=28, color=CTRLC, edgecolor="white", alpha=.9,
            label=r"moving-block permutations  $S_q(\hat u_\pi)$")
    ax.axvline(S_obs, color=RED, lw=2.4,
               label=f"observed  $S_q(\\hat u)$   (p = {pval:.3f})")
    ax.set_xlabel(r"test statistic  $S_q=(\frac{1}{T_{post}}\sum_t|\hat u_t|^q)^{1/q}$")
    ax.set_ylabel("permutations"); ax.grid(axis="x")
    ax.set_title("Permutation null distribution, H₀: no effect (θ = 0)")
    ax.legend(loc="upper right", fontsize=9, framealpha=.9)
    _save(fig, "Conformal_Figure_5.png")


def fig6(fit=None):                                   # CI by test inversion
    df, tid, obs, cf, perm, lengths, att = fit or _fit_sc()
    grid = np.arange(-3, 6, 0.05)
    ci = conformal_inf.ci_calc(y_hat=cf.copy(), y_act=obs.copy(), theta_grid=grid,
                               permutation_list_ci=perm, pre_pst_lengths_ci=lengths,
                               alpha=0.05)
    pv = np.array(ci["pvalue_list"]); lo, hi = ci["ci_interval"]
    est = float(np.mean((obs - cf)[-T_POST_SC:]))
    fig, ax = plt.subplots(figsize=(8.6, 3.9))
    ax.plot(grid, pv, color=ACCENT, lw=2.2, label=r"conformal p-value  $p(\theta)$")
    ax.axhline(0.05, color=MUTED, ls="--", lw=1.2)
    ax.annotate("α = 0.05", xy=(grid[2], 0.05), xytext=(0, 4),
                textcoords="offset points", fontsize=8.5, color=MUTED)
    ax.axvspan(lo, hi, color=ACCENT, alpha=.12,
               label=f"95% conformal CI  [{lo:.2f}, {hi:.2f}]")
    ax.axvline(est, color=ACCENT2, lw=1.8, ls=":", label=f"point est  {est:.2f}")
    ax.axvline(att, color=RED, lw=1.8, label=f"true effect  {att:.2f}")
    ax.set_xlabel(r"candidate constant effect  $\theta$")
    ax.set_ylabel("p-value"); ax.set_ylim(-.02, 1.02)
    ax.set_title("Confidence interval by test inversion")
    ax.legend(loc="upper right", fontsize=8.5, framealpha=.9)
    _save(fig, "Conformal_Figure_6.png")

In [ ]:
fig0(); fig1(); fig2(); fig3()
_scfit = _fit_sc()
fig4(_scfit); fig5(_scfit); fig6(_scfit)
print("figures written to", FIGDIR)